# IS 362 — Week 7: Movie Ratings Aggregation

**Dataset:** Five fictional reviewers rate six popular movies on a 1–5 scale. Missing entries mean the reviewer has not seen the film. These are illustrative fictional observations, not real survey responses.

**Goal:** Compare raw ratings, per-user min–max normalized ratings, and per-user z-score standardized ratings.

In [ ]:
import pandas as pd

pd.set_option("display.precision", 3)
ratings = pd.read_csv("movie_ratings.csv", index_col="Reviewer")
print("Original movie ratings (NaN = not seen):")
display(ratings)

## 1–2. Average ratings

Compute the mean of each reviewer’s observed ratings (row-wise) and the mean for each movie across reviewers who rated it (column-wise). Missing values are excluded.

In [ ]:
print("Average rating by reviewer:")
display(ratings.mean(axis=1).rename("Average rating").to_frame())
print("Average rating by movie:")
display(ratings.mean(axis=0).rename("Average rating").to_frame())

## 3. Per-user normalized ratings (min–max)

For each reviewer, normalize each observed rating as **(rating − reviewer minimum) / (reviewer maximum − reviewer minimum)**. This maps each reviewer’s ratings to the 0–1 range. Missing values remain missing. If a reviewer gives all movies the same rating, assign 0.0 to observed ratings to avoid dividing by zero.

In [ ]:
row_min = ratings.min(axis=1)
row_range = ratings.max(axis=1) - row_min
normalized = ratings.sub(row_min, axis=0).div(row_range.replace(0, float("nan")), axis=0)
normalized = normalized.where(~ratings.notna() | row_range.ne(0), 0.0)
print("Normalized ratings:")
display(normalized)
print("Average normalized rating by reviewer:")
display(normalized.mean(axis=1).rename("Average normalized rating").to_frame())
print("Average normalized rating by movie:")
display(normalized.mean(axis=0).rename("Average normalized rating").to_frame())

## 4. Interpretation

**Advantages:** Normalization reduces the effect of reviewers using different portions of the 1–5 scale. It makes it easier to compare relative preferences, even when one reviewer tends to rate movies more generously.

**Disadvantages:** It removes the original meaning of the 1–5 scores, and reviewers with very narrow rating ranges can have small differences exaggerated. Movie averages are also based on different subsets of reviewers when ratings are missing. With only a few ratings per reviewer, the normalized values can be unstable. Normalization does not solve the missing-data problem.

**Conclusion:** Use raw ratings when the absolute star score matters; use normalized ratings when relative preferences across reviewers matter. Interpret both with caution for small samples.

## 5. Extra credit — per-user standardized ratings (z-scores)

For each reviewer, compute **(rating − reviewer mean) / reviewer standard deviation**. We use the population standard deviation (`ddof=0`) for each reviewer’s observed ratings. A zero standard deviation is handled by assigning 0.0 to observed ratings.

In [ ]:
row_mean = ratings.mean(axis=1)
row_std = ratings.std(axis=1, ddof=0)
standardized = ratings.sub(row_mean, axis=0).div(row_std.replace(0, float("nan")), axis=0)
standardized = standardized.where(~ratings.notna() | row_std.ne(0), 0.0)
print("Standardized ratings:")
display(standardized)
print("Average standardized rating by reviewer:")
display(standardized.mean(axis=1).rename("Average standardized rating").to_frame())
print("Average standardized rating by movie:")
display(standardized.mean(axis=0).rename("Average standardized rating").to_frame())

## Final notes

A positive z-score means a reviewer liked a movie more than their own average; a negative score means less than their own average. Per-reviewer mean z-scores should be approximately zero (subject to floating-point rounding). This fictional example is intended to demonstrate pandas aggregation and transformations, not to estimate population-wide movie preferences.